# 🍈 Download & Extract Durian Dataset Directly to Google Drive (Google Colab)

This notebook runs on **Google Colab** to leverage Google's high-speed network infrastructure (100 - 300 MB/s):
1. **Automatically Mount your Google Drive**.
2. **Download dataset parts sequentially from Zenodo Record: 18603796**.
3. **Automatically extract** directly into Google Drive.
4. **Automatically delete `.zip` files** immediately after each part finishes extracting to save storage space.

> **Note:** Simply open this notebook on Google Colab and click **Runtime -> Run all** (or Ctrl + F9).

### Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Target directory path on Google Drive
# Default path: MyDrive/durian-ripeness-cmkd/data/raw
DRIVE_TARGET_DIR = '/content/drive/MyDrive/durian-ripeness-cmkd/data/raw'
os.makedirs(DRIVE_TARGET_DIR, exist_ok=True)

print(f'Target directory on Google Drive: {DRIVE_TARGET_DIR}')

Mounted at /content/drive
Target directory on Google Drive: /content/drive/MyDrive/durian-ripeness-cmkd/data/raw


### Step 2: Install high-speed download tool (optional)

In [ ]:
# Install aria2 for high-speed multi-threaded downloads if needed
!apt-get install -y -qq aria2
print('Download environment is ready!')

Selecting previously unselected package libcares2:amd64.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Setting up libaria2-0:amd64 (1.37.0+debian-1build3) ...
Setting up aria2 (1.37.0+debian-1build3) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

### Step 3: Automated Script to Download -> Extract to Drive -> Remove Archives

In [ ]:
import os
import sys
import json
import time
import subprocess
import urllib.request

RECORD_ID = '18603796'
ZENODO_API_URL = f'https://zenodo.org/api/records/{RECORD_ID}'
LOCAL_TEMP_DIR = '/content/temp_downloads'
os.makedirs(LOCAL_TEMP_DIR, exist_ok=True)

def format_size(size_bytes):
    if size_bytes < 1024 ** 2:
        return f'{size_bytes / 1024:.2f} KB'
    elif size_bytes < 1024 ** 3:
        return f'{size_bytes / (1024 ** 2):.2f} MB'
    else:
        return f'{size_bytes / (1024 ** 3):.2f} GB'

print(f'[*] Connecting to Zenodo API: {ZENODO_API_URL} ...')
req = urllib.request.Request(ZENODO_API_URL, headers={'User-Agent': 'Colab-Downloader/1.0'})
with urllib.request.urlopen(req) as resp:
    metadata = json.loads(resp.read().decode('utf-8'))

files = metadata.get('files', [])
print(f'Found {len(files)} files in Zenodo Record:')
for idx, f in enumerate(files, 1):
    print(f'  {idx}. {f["key"]} ({format_size(f["size"])})')

print('\n' + '=' * 60)
print('STARTING DOWNLOAD, EXTRACTION TO DRIVE, AND CLEANUP')
print('=' * 60)

for idx, f in enumerate(files, 1):
    fname = f['key']
    fsize = f['size']
    furl = f['links']['self']

    print(f'\n[{idx}/{len(files)}] Processing: {fname} ({format_size(fsize)})')

    # For lightweight files (.csv, .md), download directly to Google Drive
    if not fname.endswith('.zip'):
        dest_file = os.path.join(DRIVE_TARGET_DIR, fname)
        print(f'    -> Downloading directly to Drive: {dest_file}')
        subprocess.run(['aria2c', '-x', '16', '-s', '16', '-k', '1M', '-d', DRIVE_TARGET_DIR, '-o', fname, furl], check=True)
        continue

    # For large .zip archives:
    # 1. Download to Colab local disk (/content/temp_downloads) for maximum speed
    local_zip_path = os.path.join(LOCAL_TEMP_DIR, fname)
    print(f'    1. Fast-downloading to Colab temporary disk...')
    subprocess.run([
        'aria2c', '-x', '16', '-s', '16', '-k', '1M',
        '-d', LOCAL_TEMP_DIR, '-o', fname, furl
    ], check=True)

    # 2. Extract directly into the Google Drive directory
    print(f'    2. Extracting directly to Google Drive ({DRIVE_TARGET_DIR})...')
    subprocess.run(['unzip', '-q', '-o', local_zip_path, '-d', DRIVE_TARGET_DIR], check=True)
    print(f'    -> Finished extracting {fname} to Drive.')

    # 3. Remove zip archive immediately to free Colab disk space
    if os.path.exists(local_zip_path):
        os.remove(local_zip_path)
        print(f'    3. Deleted temporary archive {fname}.')

print('\n' + '=' * 60)
print('🎉 COMPLETED! ALL DATASET FILES SAVED TO GOOGLE DRIVE!')
print('=' * 60)

[*] Connecting to Zenodo API: https://zenodo.org/api/records/18603796 ...
Found 6 files in Zenodo Record:
  1. README.md (15.62 KB)
  2. dataset_clean_multispectral.zip (928.48 MB)
  3. dataset_clean_sound.zip (1.51 GB)
  4. dataset_clean_thermal.zip (511.49 MB)
  5. dataset_clean_rgb.zip (18.89 GB)
  6. durian_characteristics_cleaned.csv (44.85 KB)

STARTING DOWNLOAD, EXTRACTION TO DRIVE, AND CLEANUP

[1/6] Processing: README.md (15.62 KB)
    -> Downloading directly to Drive: /content/drive/MyDrive/durian-ripeness-cmkd/data/raw/README.md

[2/6] Processing: dataset_clean_multispectral.zip (928.48 MB)
    1. Fast-downloading to Colab temporary disk...
    2. Extracting directly to Google Drive (/content/drive/MyDrive/durian-ripeness-cmkd/data/raw)...
    -> Finished extracting dataset_clean_multispectral.zip to Drive.
    3. Deleted temporary archive dataset_clean_multispectral.zip.

[3/6] Processing: dataset_clean_sound.zip (1.51 GB)
    1. Fast-downloading to Colab temporary disk...


### Step 4: Verify Extracted Directories on Google Drive

In [ ]:
!ls -la "/content/drive/MyDrive/durian-ripeness-cmkd/data/raw"

total 89
drwx------ 7 root root  4096 Oct 21 15:48 .
drwx------ 3 root root  4096 Oct 21 15:28 ..
drwx------ 2 root root  4096 Oct 21 15:29 dataset_clean_multispectral
drwx------ 2 root root  4096 Feb  2  2026 dataset_clean_rgb
drwx------ 2 root root  4096 Feb  4  2026 dataset_clean_sound
drwx------ 2 root root  4096 Feb 10  2026 dataset_clean_thermal
-rw------- 1 root root 45923 Oct 21 15:48 durian_characteristics_cleaned.csv
drwx------ 3 root root  4096 Oct 21 15:29 __MACOSX
-rw------- 1 root root 15997 Oct 21 15:29 README.md


### 🍈 Step 5: Split Dataset into Consistent Train / Val / Test Splits (Cross-Modal Stratified Split)

> ⚠️ **CRITICAL PRINCIPLE ON CROSS-MODAL DATA CONSISTENCY:**
> 1. **Fruit-Level Split (Sample-level Split):** Never **split individual images or audio files randomly**. If a TOP-view image of fruit #45 lands in Train while its BOT-view image or sensor row lands in Test -> this causes severe **Data Leakage** and breaks cross-modal consistency!
> 2. **100% Synchronization Across Modalities:** All RGB images (6 views), knock audio files (`.m4a`, `.wav`), thermal images, thermal matrices, multispectral images (`.tif`), and the **feature row in `durian_characteristics_cleaned.csv`** of the same durian fruit **must belong to one single split (Train, Val, or Test)**.
> 3. **Stratified Split by `Ripeness` Label:** Ensures balanced proportions of ripeness classes (`Unripe`, `Ripe`, `Overripe`) across all 3 splits (Default: 70% Train - 15% Val - 15% Test).

> 📂 **Output Directory Structure (`data/split_data/`):**
> ```text
> data/split_data/
> ├── sample_split_mapping.csv           # Master lookup table mapping all 189 fruits to splits
> ├── train/
> │   ├── dataset_clean_rgb/              # RGB images of fruits in Train split
> │   ├── dataset_clean_sound/            # Audio files of fruits in Train split
> │   ├── dataset_clean_thermal/          # Thermal files of fruits in Train split
> │   ├── dataset_clean_multispectral/    # Multispectral files of fruits in Train split
> │   └── durian_characteristics_cleaned.csv # Sensor characteristics CSV for Train split
> ├── val/
> │   ├── dataset_clean_rgb/
> │   ├── dataset_clean_sound/
> │   ├── dataset_clean_thermal/
> │   ├── dataset_clean_multispectral/
> │   └── durian_characteristics_cleaned.csv # Sensor characteristics CSV for Val split
> └── test/
>     ├── dataset_clean_rgb/
>     ├── dataset_clean_sound/
>     ├── dataset_clean_thermal/
>     ├── dataset_clean_multispectral/
>     └── durian_characteristics_cleaned.csv # Sensor characteristics CSV for Test split
> ```

In [5]:
# ==============================================================================
# CONSISTENT MULTIMODAL DATA SPLITTING SCRIPT (CROSS-MODAL STRATIFIED SPLIT)
# Guarantees: Images, Audio, Thermal, Multispectral, and Sensor Characteristics CSV
# for the same durian fruit reside in the same split directory (Train, Val, or Test).
# ==============================================================================

import os
import re
import json
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------------------
# 1. CONFIGURE PATHS & SPLIT RATIOS
# ------------------------------------------------------------------------------
# Auto-detect runtime environment: Google Colab or Local machine
COLAB_RAW_DIR = '/content/drive/MyDrive/durian-ripeness-cmkd/data/raw'
COLAB_SPLIT_DIR = '/content/drive/MyDrive/durian-ripeness-cmkd/data/split_data/'

LOCAL_RAW_DIR = os.path.abspath('data/raw')
LOCAL_SPLIT_DIR = os.path.abspath('data/split_data')

if os.path.exists(COLAB_RAW_DIR):
    RAW_DATA_DIR = COLAB_RAW_DIR
    SPLIT_TARGET_DIR = COLAB_SPLIT_DIR
    print("[*] Detected Google Colab environment.")
else:
    # Search local candidate paths if running locally
    candidates = [
        LOCAL_RAW_DIR,
        os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data', 'raw'),
        '../../../data/raw',
        './'
    ]
    RAW_DATA_DIR = next((p for p in candidates if os.path.exists(os.path.join(p, 'durian_characteristics_cleaned.csv'))), LOCAL_RAW_DIR)
    SPLIT_TARGET_DIR = os.path.abspath(os.path.join(RAW_DATA_DIR, '..', 'split_data'))
    print("[*] Detected Local environment.")

print(f"Raw data directory (RAW)        : {RAW_DATA_DIR}")
print(f"Output directory (SPLIT_DATA)   : {SPLIT_TARGET_DIR}")

# Split ratios: 70% Train, 15% Val, 15% Test (adjustable as needed)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

CSV_FILENAME = 'durian_characteristics_cleaned.csv'
CSV_PATH = os.path.join(RAW_DATA_DIR, CSV_FILENAME)

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Characteristics CSV file not found at: {CSV_PATH}. Please verify the download step!")

# ------------------------------------------------------------------------------
# 2. LOAD CHARACTERISTICS TABLE AND RUN FRUIT-LEVEL STRATIFIED SPLIT
# ------------------------------------------------------------------------------
df_raw = pd.read_csv(CSV_PATH)
print(f"\n[*] Loaded characteristics table: {len(df_raw)} durian fruits.")

# Extract integer sample_id (1 to 189) from Replicate column (e.g., 'R1' -> 1)
df_raw['sample_id'] = df_raw['Replicate'].astype(str).str.extract(r'(\d+)')[0].astype(int)

# Lookup table mapping durian Code -> sample_id
code_to_id = dict(zip(df_raw['Code'], df_raw['sample_id']))

# Stratified Split by 'Ripeness' label
# Step a: Separate Train (70%) and Temporary pool (30%)
temp_ratio = VAL_RATIO + TEST_RATIO
train_df, temp_df = train_test_split(
    df_raw,
    test_size=temp_ratio,
    stratify=df_raw['Ripeness'],
    random_state=RANDOM_SEED
)

# Step b: Split Temporary pool (30%) evenly into Val (15%) and Test (15%)
val_relative_ratio = VAL_RATIO / temp_ratio
val_df, test_df = train_test_split(
    temp_df,
    test_size=(1.0 - val_relative_ratio),
    stratify=temp_df['Ripeness'],
    random_state=RANDOM_SEED
)

# Collect fruit ID sets for each split
train_sample_ids = set(train_df['sample_id'].tolist())
val_sample_ids = set(val_df['sample_id'].tolist())
test_sample_ids = set(test_df['sample_id'].tolist())

# Check mutual exclusivity (ensure zero overlapping fruits across splits)
assert len(train_sample_ids & val_sample_ids) == 0, "Error: Overlap between Train and Val!"
assert len(train_sample_ids & test_sample_ids) == 0, "Error: Overlap between Train and Test!"
assert len(val_sample_ids & test_sample_ids) == 0, "Error: Overlap between Val and Test!"
assert len(train_sample_ids) + len(val_sample_ids) + len(test_sample_ids) == len(df_raw), "Error: Total fruit count mismatch!"

print(f"\n{'=' * 65}")
print(" FRUIT & RIPENESS LABEL DISTRIBUTION ACROSS SPLITS ")
print(f"{'=' * 65}")
split_dfs = {'train': train_df, 'val': val_df, 'test': test_df}
split_ids = {'train': train_sample_ids, 'val': val_sample_ids, 'test': test_sample_ids}

for s_name, s_df in split_dfs.items():
    counts = s_df['Ripeness'].value_counts().to_dict()
    print(f"* Split {s_name.upper():<5}: {len(s_df):>3} fruits | Unripe: {counts.get('Unripe', 0):>2}, Ripe: {counts.get('Ripe', 0):>2}, Overripe: {counts.get('Overripe', 0):>2}")
print(f"{'=' * 65}")

# ------------------------------------------------------------------------------
# 3. EXTRACT SAMPLE_ID FROM MULTIMODAL FILENAMES
# ------------------------------------------------------------------------------
def extract_sample_id_from_filename(filename, code_map):
    """Identify sample_id (1..189) for any filename format in the dataset."""
    # Pattern 1: durian_001_..., durian_045_... (RGB, Thermal JPG/CSV, Multispectral TIF)
    match_durian = re.search(r'durian_0*(\d+)', filename, re.IGNORECASE)
    if match_durian:
        return int(match_durian.group(1))

    # Pattern 2: 45-R.m4a, 1-U.m4a (Acoustic knock iPhone)
    match_iphone = re.match(r'^(\d+)[-_]', filename)
    if match_iphone:
        return int(match_iphone.group(1))

    # Pattern 3: IM_CA_UN_1.wav, MA_CB_R_45.wav (Acoustic knock Android)
    match_android = re.search(r'_(\d+)\.(wav|m4a|mp3)$', filename, re.IGNORECASE)
    if match_android:
        return int(match_android.group(1))

    # Pattern 4: Direct lookup by fruit Code
    base_name = os.path.splitext(filename)[0]
    if base_name in code_map:
        return code_map[base_name]

    return None

# ------------------------------------------------------------------------------
# 4. CREATE DIRECTORY STRUCTURE AND COPY SYNCHRONIZED DATA
# ------------------------------------------------------------------------------
MODALITY_FOLDERS = [
    'dataset_clean_rgb',
    'dataset_clean_sound',
    'dataset_clean_thermal',
    'dataset_clean_multispectral'
]

# Create train, val, test directories and modality subdirectories
for s_name in ['train', 'val', 'test']:
    s_dir = os.path.join(SPLIT_TARGET_DIR, s_name)
    os.makedirs(s_dir, exist_ok=True)
    for m_folder in MODALITY_FOLDERS:
        os.makedirs(os.path.join(s_dir, m_folder), exist_ok=True)

# A. Save 3 characteristics CSV files into train/, val/, test/ (DROPPING temporary 'sample_id')
print("\n[*] Saving characteristics CSV files into train, val, test directories...")
for s_name, s_df in split_dfs.items():
    out_csv = os.path.join(SPLIT_TARGET_DIR, s_name, CSV_FILENAME)
    s_df.drop(columns=['sample_id']).to_csv(out_csv, index=False)
    print(f"    -> Saved: {s_name}/{CSV_FILENAME} ({len(s_df)} rows)")

# Save consolidated lookup table sample_split_mapping.csv for future reference
mapping_rows = []
for s_name, s_df in split_dfs.items():
    for _, row in s_df.iterrows():
        mapping_rows.append({
            'sample_id': row['sample_id'],
            'Replicate': row['Replicate'],
            'Code': row['Code'],
            'Ripeness': row['Ripeness'],
            'Maturity_Stage': row.get('Maturity_Stage', ''),
            'Split': s_name
        })
mapping_df = pd.DataFrame(mapping_rows).sort_values('sample_id')
mapping_path = os.path.join(SPLIT_TARGET_DIR, 'sample_split_mapping.csv')
mapping_df.to_csv(mapping_path, index=False)
print(f"    -> Saved master lookup table: sample_split_mapping.csv")

# B. Categorize and copy files for each Modality
print("\n[*] Sorting and copying multimodal files into train, val, test...")
copy_stats = {s: {m: 0 for m in MODALITY_FOLDERS} for s in ['train', 'val', 'test']}

for m_folder in MODALITY_FOLDERS:
    src_mod_dir = os.path.join(RAW_DATA_DIR, m_folder)
    if not os.path.exists(src_mod_dir):
        print(f"    [!] Skipping '{m_folder}' (Not found in data/raw)")
        continue

    all_files = [f for f in os.listdir(src_mod_dir) if not f.startswith('.')]
    if not all_files:
        print(f"    [!] Skipping '{m_folder}' (Empty directory)")
        continue

    # Separately handle metadata.csv inside modality folder if present
    meta_csv_path = os.path.join(src_mod_dir, 'metadata.csv')
    has_meta_csv = os.path.exists(meta_csv_path)
    if has_meta_csv:
        try:
            meta_df = pd.read_csv(meta_csv_path)
            # Identify ID column in metadata.csv
            id_col = next((c for c in ['sample_num', 'sample_id', 'sample', 'Code'] if c in meta_df.columns), None)
            if id_col:
                meta_df['norm_id'] = meta_df[id_col].astype(str).str.extract(r'(\d+)')[0].astype(float)
                for s_name, s_ids in split_ids.items():
                    sub_meta = meta_df[meta_df['norm_id'].isin(s_ids)].drop(columns=['norm_id'])
                    sub_meta.to_csv(os.path.join(SPLIT_TARGET_DIR, s_name, m_folder, 'metadata.csv'), index=False)
        except Exception as e:
            print(f"    [!] Could not split metadata.csv for {m_folder}: {e}")

    # Copy data files (images, audio, thermal matrices, tif)
    copied_mod = 0
    for fname in all_files:
        if fname == 'metadata.csv':
            continue

        sample_id = extract_sample_id_from_filename(fname, code_to_id)
        if sample_id is None:
            continue

        # Determine target split (train, val, test)
        target_split = None
        if sample_id in train_sample_ids:
            target_split = 'train'
        elif sample_id in val_sample_ids:
            target_split = 'val'
        elif sample_id in test_sample_ids:
            target_split = 'test'

        if target_split:
            src_file = os.path.join(src_mod_dir, fname)
            dst_file = os.path.join(SPLIT_TARGET_DIR, target_split, m_folder, fname)
            shutil.copy2(src_file, dst_file)
            copy_stats[target_split][m_folder] += 1
            copied_mod += 1

    print(f"    -> [{m_folder}]: Finished splitting {copied_mod} files (Train: {copy_stats['train'][m_folder]}, Val: {copy_stats['val'][m_folder]}, Test: {copy_stats['test'][m_folder]})")

# ------------------------------------------------------------------------------
# 5. CONSISTENCY VERIFICATION AND SUMMARY REPORT
# ------------------------------------------------------------------------------
print(f"\n{'=' * 75}")
print(" MULTIMODAL DATA SPLITTING SUMMARY REPORT ")
print(f"{'=' * 75}")
print(f"{'Split':<12} | {'Fruits':<8} | {'CSV Rows':<10} | {'RGB Images':<10} | {'Audio':<10} | {'Thermal':<10} | {'Spectrum':<10}")
print("-" * 75)
for s in ['train', 'val', 'test']:
    csv_rows = len(split_dfs[s])
    rgb_count = copy_stats[s]['dataset_clean_rgb']
    snd_count = copy_stats[s]['dataset_clean_sound']
    thm_count = copy_stats[s]['dataset_clean_thermal']
    msp_count = copy_stats[s]['dataset_clean_multispectral']
    print(f"{s.upper():<12} | {len(split_ids[s]):<8} | {csv_rows:<10} | {rgb_count:<10} | {snd_count:<10} | {thm_count:<10} | {msp_count:<10}")
print(f"{'=' * 75}")
print("✅ COMPLETED! Data has been split consistently across RGB, audio, thermal, multispectral, and sensor CSV!")


[*] Detected Google Colab environment.
Raw data directory (RAW)        : /content/drive/MyDrive/durian-ripeness-cmkd/data/raw
Output directory (SPLIT_DATA)   : /content/drive/MyDrive/durian-ripeness-cmkd/data/split_data/

[*] Loaded characteristics table: 189 durian fruits.

 FRUIT & RIPENESS LABEL DISTRIBUTION ACROSS SPLITS 
* Split TRAIN: 132 fruits | Unripe: 44, Ripe: 44, Overripe: 44
* Split VAL  :  28 fruits | Unripe:  9, Ripe:  9, Overripe: 10
* Split TEST :  29 fruits | Unripe: 10, Ripe: 10, Overripe:  9

[*] Saving characteristics CSV files into train, val, test directories...
    -> Saved: train/durian_characteristics_cleaned.csv (132 rows)
    -> Saved: val/durian_characteristics_cleaned.csv (28 rows)
    -> Saved: test/durian_characteristics_cleaned.csv (29 rows)
    -> Saved master lookup table: sample_split_mapping.csv

[*] Sorting and copying multimodal files into train, val, test...
    -> [dataset_clean_rgb]: Finished splitting 2077 files (Train: 1497, Val: 282, Test: 2

### 🔍 Step 6: Inspect Directory Structure and Label Distribution After Splitting

In [6]:
# Inspect file counts and CSV summaries for each split in data/split_data/
import os
import pandas as pd

target_dir = SPLIT_TARGET_DIR if 'SPLIT_TARGET_DIR' in locals() else '/content/drive/MyDrive/durian-ripeness-cmkd/data/split_data/'
print(f"Inspecting directory: {target_dir}\n")

for split in ['train', 'val', 'test']:
    split_folder = os.path.join(target_dir, split)
    csv_file = os.path.join(split_folder, 'durian_characteristics_cleaned.csv')
    print(f"📂 SPLIT [{split.upper()}]:")
    if os.path.exists(csv_file):
        df_s = pd.read_csv(csv_file)
        print(f"   ├── durian_characteristics_cleaned.csv: {len(df_s)} durian fruit rows")
        print(f"   │   └── Ripeness distribution: {df_s['Ripeness'].value_counts().to_dict()}")
    else:
        print("   ├── durian_characteristics_cleaned.csv: ⚠️ Missing")

    for mod in ['dataset_clean_rgb', 'dataset_clean_sound', 'dataset_clean_thermal', 'dataset_clean_multispectral']:
        mod_p = os.path.join(split_folder, mod)
        if os.path.exists(mod_p):
            cnt = len([f for f in os.listdir(mod_p) if not f.startswith('.')])
            print(f"   ├── {mod:<30}: {cnt} files")
        else:
            print(f"   ├── {mod:<30}: ⚠️ Not created")
    print("-" * 60)


Inspecting directory: /content/drive/MyDrive/durian-ripeness-cmkd/data/split_data/

📂 SPLIT [TRAIN]:
   ├── durian_characteristics_cleaned.csv: 132 durian fruit rows
   │   └── Ripeness distribution: {'Overripe': 44, 'Ripe': 44, 'Unripe': 44}
   ├── dataset_clean_rgb             : 1498 files
   ├── dataset_clean_sound           : 407 files
   ├── dataset_clean_thermal         : 2493 files
   ├── dataset_clean_multispectral   : 1681 files
------------------------------------------------------------
📂 SPLIT [VAL]:
   ├── durian_characteristics_cleaned.csv: 28 durian fruit rows
   │   └── Ripeness distribution: {'Overripe': 10, 'Unripe': 9, 'Ripe': 9}
   ├── dataset_clean_rgb             : 283 files
   ├── dataset_clean_sound           : 79 files
   ├── dataset_clean_thermal         : 539 files
   ├── dataset_clean_multispectral   : 324 files
------------------------------------------------------------
📂 SPLIT [TEST]:
   ├── durian_characteristics_cleaned.csv: 29 durian fruit rows
   │   